# EDA — WaRP-C Waste Classification Dataset

El Mehdi Ziate

## Imports

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / 'Pipeline_').exists() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f'Project root  : {root}')
print(f'sys.path[0]   : {sys.path[0]}')

Project root  : /home/el-mehdi-ziate/Desktop/Waste-Classification
sys.path[0]   : /home/el-mehdi-ziate/Desktop/Waste-Classification


In [2]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from Pipeline_.eda import EDAModule

## Initialise the EDA module

In [3]:
# here you can change the paths based on where you have saved the data

eda = EDAModule(
    data_root   = root / 'Dataset/raw/Warp-C',
    figures_dir = root / 'Dataset/figures',
    stats_file  = root / 'Dataset/dataset_stats.json',
    seed        = 42,
)

print(f'\nClasses found : {eda.classes}')
print(f'\nParent classes found: {eda.parent_classes}')

[EDAModule] Dataset loaded from: /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/raw/Warp-C
[EDAModule] Train classes : 28
[EDAModule] Test  classes : 28
[EDAModule] Figures → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures

Classes found : ['bottle-blue', 'bottle-blue-full', 'bottle-blue5l', 'bottle-blue5l-full', 'bottle-dark', 'bottle-dark-full', 'bottle-green', 'bottle-green-full', 'bottle-milk', 'bottle-milk-full', 'bottle-multicolor', 'bottle-multicolorv-full', 'bottle-oil', 'bottle-oil-full', 'bottle-transp', 'bottle-transp-full', 'bottle-yogurt', 'canister', 'cans', 'detergent-box', 'detergent-color', 'detergent-transparent', 'detergent-white', 'glass-dark', 'glass-green', 'glass-transp', 'juice-cardboard', 'milk-cardboard']

Parent classes found: defaultdict(<class 'set'>, {'bottle': {'bottle-oil', 'bottle-transp', 'glass-transp', 'glass-dark', 'bottle-transp-full', 'bottle-dark', 'bottle-green', 'bottle-blue5l-full', 'bottle-multicolorv-full', 'bott

## Class Distribution

In [4]:
dist_stats = eda.plot_class_distribution()

  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/01_class_distribution_bottle.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/01_class_distribution_canister.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/01_class_distribution_cans.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/01_class_distribution_cardboard.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/01_class_distribution_detergent.png

  Imbalance ratio : 59.67:1
  Majority class  : bottle-transp (1432 images)
  Minority class  : bottle-oil-full (24 images)
  Total train     : 8823
  Total test      : 1551
  → Severe imbalance. Consider oversampling minority classes too.


In [5]:
import pandas as pd

df_dist = pd.DataFrame({
    'Class'      : dist_stats['class_names'],
    'Train count': dist_stats['train_counts'],
    'Test count' : dist_stats['test_counts'],
}).set_index('Class')

df_dist['Train %'] = (df_dist['Train count'] / dist_stats['total_train'] * 100).round(2)
df_dist['Test %']  = (df_dist['Test count']  / dist_stats['total_test']  * 100).round(2)

print(f"Imbalance ratio: {dist_stats['imbalance_ratio']}:1\n")
df_dist.sort_values('Train count', ascending=False)

Imbalance ratio: 59.67:1



,Train count,Test count,Train %,Test %
Class,,,,
bottle-transp,1432,234,16.23,15.09
bottle-blue,634,104,7.19,6.71
cans,562,98,6.37,6.32
bottle-dark,533,95,6.04,6.13
bottle-transp-full,528,92,5.98,5.93
bottle-green,466,74,5.28,4.77
bottle-blue5l,413,72,4.68,4.64
milk-cardboard,390,94,4.42,6.06
bottle-milk,347,57,3.93,3.68


## Sample Grid

In [6]:
eda.plot_sample_grid(n_samples=5)

  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/02_sample_grid_bottle.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/02_sample_grid_canister.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/02_sample_grid_cans.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/02_sample_grid_cardboard.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/02_sample_grid_detergent.png


## Image Size & Aspect Ratio

**What this shows:**  
Histograms of width, height, and aspect ratio (W/H) across all images.


In [7]:
size_stats = eda.analyze_image_sizes()

  Scanning image sizes: 100%|██████████| 10374/10374 [00:00<00:00, 19732.79it/s]


  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/03_image_sizes.png

  Width  median=162.0px   range=[40, 703]
  Height median=154.0px   range=[35, 668]
  Aspect median=1.1   range=[0.20, 5.20]

  Recommended resize target : 192px  (75th-pct of shorter side, rounded to nearest 32)


## Pixel Statistics (Normalisation)

**What this shows:**  
Per-channel (R, G, B) mean and standard deviation for WaRP-C, compared to standard ImageNet statistics.

**Why this matters:**  
Every pretrained model expects normalised inputs. We compute WaRP-C-specific stats and compare  
to ImageNet. If the difference is small (< 0.05), we can safely use ImageNet stats with pretrained  
backbones. If large, we should use WaRP-C stats. since we are going to use YOLOV8, ResNet we need this so we can decide how to move on



In [8]:
pixel_stats = eda.compute_pixel_stats(sample_size=1500)

  Computing pixel stats: 100%|██████████| 1500/1500 [00:03<00:00, 384.66it/s]


  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/04_pixel_stats.png

  WaRP-C mean  : [np.float64(0.337), np.float64(0.344), np.float64(0.35)]
  WaRP-C std   : [np.float64(0.216), np.float64(0.209), np.float64(0.218)]
  ImageNet mean: [0.485, 0.456, 0.406]
  ImageNet std : [0.229, 0.224, 0.225]
  Mean deviation from ImageNet: 0.1051
  → Use WaRP-C stats — notable difference from ImageNet domain.


In [9]:
mean = pixel_stats['warp_mean']
std  = pixel_stats['warp_std']
imn_mean = pixel_stats['imagenet_mean']
imn_std  = pixel_stats['imagenet_std']

print('── Normalisation to use in transforms.py ──────────────────────')
print(f'# WaRP-C specific:')
print(f'transforms.Normalize(mean={[round(v,3) for v in mean]}, std={[round(v,3) for v in std]})')
print(f'\n# ImageNet (safe default for pretrained models):')
print(f'transforms.Normalize(mean={imn_mean}, std={imn_std})')
print(f'\nRecommendation: {pixel_stats["recommendation"]}')

── Normalisation to use in transforms.py ──────────────────────
# WaRP-C specific:
transforms.Normalize(mean=[0.337, 0.344, 0.35], std=[0.216, 0.209, 0.218])

# ImageNet (safe default for pretrained models):
transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

Recommendation: Use WaRP-C stats — notable difference from ImageNet domain.


## Train / Test Split Comparison

**What this shows:**  
The percentage share of each class in train vs test.

**What to look for:**  
Identical bars = perfect stratified split.  
Large deviations (> 2 pp) = the split is not stratified  we need to flag this in our report.

In [10]:
split_stats = eda.plot_train_test_comparison()

  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/05_train_test_split_bottle.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/05_train_test_split_canister.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/05_train_test_split_cans.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/05_train_test_split_cardboard.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/05_train_test_split_detergent.png

  Max proportion deviation : 1.64%  (class 'milk-cardboard')
  → Split looks well-stratified.


## Duplicate / Data Leakage Check

**What this shows:**  
Whether any filenames appear in both train and test.

**Why this matters:**  
If the same image is in both splits, the model has already seen test images during training which means that the test accuracy is artificially inflated and results are meaningless.

In [11]:
dup_stats = eda.check_duplicates()


  Train unique filenames : 3360
  Test  unique filenames : 667
  Overlap count          : 18
  ⚠  WARNING: 18 duplicate filenames — potential data leakage!
  Examples: ['Monitoring_photo2_04-Mar_00-47-42_01.jpg', 'POSAD_1_11-Sep_11-49-05_01.jpg', 'POSAD_1_12-Sep_11-09-35_01.jpg', 'POSAD_1_12-Sep_11-09-35_02.jpg', 'POSAD_1_12-Sep_11-09-35_03.jpg']


## Per-Class Brightness Analysis

**What this shows:**  
Mean luminance (greyscale brightness) per class.

**What to look for:**  
- Dark classes (dark glass, dark bottles) vs bright classes (white detergent, milk cartons)  
- High brightness standard deviation across classes → use **strong** `ColorJitter` augmentation  
- Two visually similar classes that differ mainly in brightness → the model may learn  
  brightness as a shortcut, which won't generalise to new lighting conditions

In [12]:
brightness_stats = eda.plot_brightness_per_class(n_per_class=60)

  Brightness per class:   0%|          | 0/28 [00:00<?, ?it/s]

  Brightness per class: 100%|██████████| 28/28 [00:00<00:00, 46.57it/s]
/home/el-mehdi-ziate/Desktop/Waste-Classification/Pipeline_/eda.py:686: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(group_classes, rotation=30, ha="right", fontsize=9)


  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/06_brightness_bottle.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/06_brightness_canister.png


/home/el-mehdi-ziate/Desktop/Waste-Classification/Pipeline_/eda.py:686: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(group_classes, rotation=30, ha="right", fontsize=9)
/home/el-mehdi-ziate/Desktop/Waste-Classification/Pipeline_/eda.py:686: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(group_classes, rotation=30, ha="right", fontsize=9)


  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/06_brightness_cans.png
  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/06_brightness_cardboard.png


/home/el-mehdi-ziate/Desktop/Waste-Classification/Pipeline_/eda.py:686: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(group_classes, rotation=30, ha="right", fontsize=9)
/home/el-mehdi-ziate/Desktop/Waste-Classification/Pipeline_/eda.py:686: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(group_classes, rotation=30, ha="right", fontsize=9)


  ✓ Saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/06_brightness_detergent.png

  Darkest class  : 'glass-dark'  (0.213)
  Brightest class: 'detergent-white' (0.481)
  Brightness std across classes: 0.066
  → Moderate brightness variance — standard ColorJitter is fine.


## Save Stats & Final Summary

In [13]:
eda.save_stats()
eda.summary()


  ✓ Stats saved → /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/dataset_stats.json

  EDA COMPLETE — WaRP-C Summary
  Total training images  : 8823
  Total test images      : 1551
  Number of classes      : 28
  Class imbalance ratio  : 59.67:1
  Median image size      : 162.0 × 154.0 px
  Aspect ratio (median)  : 1.1
  Recommended resize     : 192 px
  WaRP-C mean (R,G,B)    : [0.337, 0.344, 0.35]
  WaRP-C std  (R,G,B)    : [0.216, 0.209, 0.218]
  Normalisation advice   : Use WaRP-C stats — notable difference from ImageNet domain.
  Train/test leakage     : ⚠ Leakage detected

  Figures saved to : /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/figures/
  Stats saved to   : /home/el-mehdi-ziate/Desktop/Waste-Classification/Dataset/dataset_stats.json

  Next step → Pipeline_/transforms.py


## Key Decisions for the Pipeline

In [14]:
print('  KEY DECISIONS')
print()
print(f"Resize target        : {size_stats.get('recommended_resize', '?')} px")
print(f"Normalisation mean   : {[round(v,3) for v in pixel_stats['warp_mean']]}")
print(f"Normalisation std    : {[round(v,3) for v in pixel_stats['warp_std']]}")
print(f"Imbalance ratio      : {dist_stats['imbalance_ratio']}:1")

ratio = dist_stats['imbalance_ratio']
if ratio < 5:
    strategy = 'Weighted CrossEntropyLoss only'
elif ratio < 50:
    strategy = 'WeightedRandomSampler + Weighted CrossEntropyLoss'
else:
    strategy = 'WeightedRandomSampler + Weighted CrossEntropyLoss + minority oversampling'

bstd = brightness_stats.get('dataset_std', 0)
jitter = 'strong (0.4)' if bstd > 0.08 else 'standard (0.2)'

print(f"Imbalance strategy   : {strategy}")
print(f"ColorJitter strength : {jitter}  (brightness std = {bstd:.3f})")
print(f"Data leakage         : {'NONE ✓' if dup_stats['overlap_count'] == 0 else '⚠ CHECK'}")
print()

  KEY DECISIONS

Resize target        : 192 px
Normalisation mean   : [0.337, 0.344, 0.35]
Normalisation std    : [0.216, 0.209, 0.218]
Imbalance ratio      : 59.67:1
Imbalance strategy   : WeightedRandomSampler + Weighted CrossEntropyLoss + minority oversampling
ColorJitter strength : standard (0.2)  (brightness std = 0.066)
Data leakage         : ⚠ CHECK

